In [ ]:
"""
Collect CLIP vision per-layer activations (layer outputs = "latter residual stream")
using Hugging Face Transformers (no hooks).

Dataset: Hugging Face Datasets (LAION-derived subset), streamed.

Output: out_dir/batch_000000.npz with:
  - ids: (B,)
  - layer00: (B, T, D) or (B*T, D) if --flatten
  - layer01: ...
and out_dir/meta.json
"""


In [ ]:
from pathlib import Path
from dataclasses import dataclass
import json

import numpy as np
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import CLIPVisionModel, CLIPImageProcessor
from huggingface_hub import snapshot_download


/home/b5bq/pu22650.b5bq/miniforge3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def save_npz(path, ids, layer_arrays):
    payload = {"ids": np.asarray(ids, dtype=np.int64)}
    for i, arr in enumerate(layer_arrays):
        payload[f"layer{i:02d}"] = arr
    np.savez_compressed(path, **payload)

In [ ]:
@dataclass
class CFG:
    out_dir: Path | None = None
    model_id: str = "openai/clip-vit-base-patch32"
    cache_dir: Path = Path(r"/scratch/b5bq/pu22650.b5bq/hf_cache")
    dataset: str = "opendiffusionai/laion2b-squareish-1024px"
    split: str = "train"
    image_col: str = "image"
    batch_size: int =64
    max_samples: int = 0 # 0 = unlimited
    device: str = "cuda"
    tokens: str = "patches" # choices=["patches", "cls", "all"]
    patch_sample: int = 0 # "0 = keep all selected tokens"
    flatten: bool = False # save as (B*T, D) instead of (B, T, D)"
    dtype: str = "fp16" # choices=["fp16", "fp32"]

In [6]:
args = CFG()
args.out_dir.mkdir(parents=True, exist_ok=True)
save_dtype = np.float16 if args.dtype == "fp16" else np.float32
device = args.device

In [9]:




model_path = snapshot_download(repo_id=args.model_id, cache_dir=args.cache_dir, local_files_only=True) # Load the model and tokenizer from the downloaded

In [ ]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained(model_path).eval().to(device)
processor = CLIPProcessor.from_pretrained(model_path)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
from vasae.models.factory import BlackBoxModelConfig


blackbox_model_cfg = BlackBoxModelConfig(
    name="clip",
    dir=Path(r"/scratch/b5bq/pu22650.b5bq/VASAE_out/BlackBoxModels/clip"),
)

blackbox_model_cfg.dir.mkdir(parents=True, exist_ok=True)
torch.save(model.text_model.embeddings.token_embedding, blackbox_model_cfg.dir / "emb.pth") # Embedding(49408, 512)

In [ ]:
ds = load_dataset(args.dataset, split=args.split, streaming=True)

In [ ]:
meta = {
    "model_id": args.model_id,
    "dataset": args.dataset,
    "split": args.split,
    "image_col": args.image_col,
    "tokens": args.tokens,
    "patch_sample": args.patch_sample,
    "flatten": bool(args.flatten),
    "dtype": args.dtype,
    "note": "Per-layer activations are outputs.hidden_states[1:] (layer outputs).",
}
with open(os.path.join(args.out_dir, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

In [ ]:

batch_imgs, batch_ids = [], []
seen, batch_idx = 0, 0

it = iter(ds)
pbar = tqdm(total=args.max_samples if args.max_samples else None, desc="collect", unit="img")

while True:
    if args.max_samples and seen >= args.max_samples:
        break

    try:
        ex = next(it)
    except StopIteration:
        break

    img = ex.get(args.image_col, None)
    if img is None:
        continue

    batch_imgs.append(img)
    batch_ids.append(seen)
    seen += 1

    if len(batch_imgs) < args.batch_size and not (args.max_samples and seen >= args.max_samples):
        pbar.update(1)
        continue

    # Process batch
    inputs = processor(images=batch_imgs, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    with torch.no_grad():
        if args.amp and device.type == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(pixel_values=pixel_values, output_hidden_states=True, return_dict=True)
        else:
            out = model(pixel_values=pixel_values, output_hidden_states=True, return_dict=True)

    # hidden_states: (embeddings output) + (each layer output)  [oai_citation:4‡Hugging Face](https://huggingface.co/docs/transformers/en/model_doc/clip?utm_source=chatgpt.com)
    hs = out.hidden_states[1:]  # per-layer outputs (what you want for "residual stream")

    layer_arrays = []
    for x in hs:
        # x: (B, seq_len, D)
        if args.tokens == "cls":
            x = x[:, :1, :]
        elif args.tokens == "patches":
            x = x[:, 1:, :]
        else:
            pass

        if args.patch_sample and x.shape[1] > args.patch_sample:
            # sample same token indices for the whole batch (simple + fast)
            idx = torch.randperm(x.shape[1], device=x.device)[: args.patch_sample]
            x = x.index_select(1, idx)

        if args.flatten:
            x = x.reshape(-1, x.shape[-1])

        layer_arrays.append(x.detach().cpu().numpy().astype(save_dtype, copy=False))

    out_path = os.path.join(args.out_dir, f"batch_{batch_idx:06d}.npz")
    save_npz(out_path, batch_ids, layer_arrays)

    batch_idx += 1
    batch_imgs, batch_ids = [], []
    pbar.update(0)  # already updated per-sample above

pbar.close()
print(f"Saved {batch_idx} batches to {args.out_dir}")

